# Full Sweep Demo

Demonstrates the complete measurement workflow described in the top-level
CLAUDE.md: configure all four instruments, sweep the Avtech pulse voltage
over a user-defined range while recording (voltage, current, lock-in
signal) via `measurement.sweep.run_voltage_sweep`, save the result with
`measurement.data_io.save_sweep_result`, and plot I-V and
optical-intensity-vs-voltage curves with `measurement.plotting`.

This notebook runs the sweep and saves its result in a single cell:

- **Live plotting**: `run_voltage_sweep`'s `on_step` callback drives a
  `clear_output(wait=True)`-based live-updating I-V/intensity plot as the
  sweep progresses.
- **Interrupt safety**: if you interrupt the sweep cell (Jupyter's
  "Interrupt Kernel"), `run_voltage_sweep` catches it internally, ramps
  the Avtech down to 0 V and disables its output (leaving the QDac
  trigger train, Rigol, and MFLI connected and running), and returns
  normally with whatever data was collected so far -- tagged
  `status="interrupted"` -- rather than raising. A safety cutoff
  (`SweepConfig.max_runtime_s`, default 1000 s) does the same if the sweep
  is left running unattended for too long.
- **Always a result**: whether the sweep completes normally, is
  interrupted, or times out, the same cell always ends with a saved
  `sweep.csv` + `metadata.json` pair in a timestamped folder under
  `data/`.

**No real instruments are connected to this development machine.** If any
instrument fails to connect, this notebook falls back to clearly-labeled
synthetic `dummy_data` (built to loosely resemble a laser-like I-V/optical
threshold curve) so the saving and plotting logic can still be
demonstrated end-to-end. On the lab computer, with all four instruments
connected, the same cells should run the real sweep instead.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root (containing `instruments/`) is importable,
# whether this notebook is run from the repo root or from within notebooks/.
project_root = Path.cwd()
if not (project_root / "instruments").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

from instruments.qdac import QDac
from instruments.avtech import Avtech
from instruments.rigol import Rigol
from instruments.mfli import MFLI
from instruments import config
from measurement.sweep import run_voltage_sweep, SweepConfig, SweepResult
from measurement import data_io, plotting

## Rigol channel offsets

Edit these before each run if a channel's signal needs to be recentered
on the scope display (e.g. after changing the DUT or wiring). Keyed by
the same channel-role names as `config.DEFAULT_CHANNEL_ROLES`; defaults to
`config.DEFAULT_CHANNEL_OFFSETS_V` (all 0 V).

In [ ]:
rigol_channel_offsets_v = dict(config.DEFAULT_CHANNEL_OFFSETS_V)
# rigol_channel_offsets_v["voltage"] = 0.5  # example override
print("Rigol channel offsets (V):", rigol_channel_offsets_v)

## Instantiate and configure all four instruments

In [ ]:
qdac = QDac()
avtech = Avtech()
rigol = Rigol()
mfli = MFLI()

instruments_connected = False

try:
    # QDac: trigger/gate signal generator that synchronizes the Avtech
    # pulse and the MFLI lock-in reference (see instruments/qdac.py).
    qdac.connect()
    qdac.channel_init(1)
    qdac.channel_init(2)
    gate_voltage = 5.0
    qdac.configure_square_wave(
        1,
        frequency=200,
        span=gate_voltage,
        offset=gate_voltage / 2,
        trigger_source=1,
    )
    qdac.configure_square_wave(
        2,
        frequency=2000,
        span=gate_voltage,
        offset=gate_voltage / 2,
        trigger_source=1,
        delay=0.0125e-3,
        duty_cycle=50,
    )
    qdac.start_square_wave(1)
    qdac.start_square_wave(2)
    qdac.fire_internal_trigger(1)

    # Avtech: swept pulse voltage source, externally triggered by the QDac.
    avtech.connect()
    avtech.remote()
    avtech.set_trigger("EXT")
    avtech.set_pulse_width(80e-6)
    avtech.output_on()

    # Rigol: reads back DUT voltage/current waveforms for each pulse.
    rigol.connect()
    rigol.configure_timebase(1e-5, offset=50e-6)
    rigol.configure_channel(
        config.DEFAULT_CHANNEL_ROLES["voltage"], 2.0, offset=rigol_channel_offsets_v["voltage"]
    )
    rigol.configure_channel(
        config.DEFAULT_CHANNEL_ROLES["current"], 2.0, offset=rigol_channel_offsets_v["current"]
    )
    rigol.configure_channel(
        config.DEFAULT_CHANNEL_ROLES["trigger"], 2.0, offset=rigol_channel_offsets_v["trigger"]
    )
    rigol.configure_trigger(
        config.DEFAULT_CHANNEL_ROLES["trigger"], level=2.0, slope="POSITIVE"
    )

    # MFLI: reads the lock-in signal corresponding to optical intensity.
    mfli.connect()
    mfli.apply_default_configuration()

    instruments_connected = True
    print("All four instruments connected and configured.")
except Exception as exc:
    print(f"Hardware not available: {exc}")
    print("Falling back to synthetic dummy_data later in this notebook.")

## Define the voltage sweep range

In [ ]:
voltages = np.linspace(2.0, 10.0, 5)
print("Sweep voltages (V):", voltages)

## Sweep settings

Edit these before each run.

In [ ]:
mfli_n_samples = 10  # MFLI samples averaged per step -- raise this if the lock-in signal is noisy
mfli_delay = 0.05  # seconds between successive MFLI samples within one averaged read
settle_time = 1.0  # seconds to wait around the Rigol single-shot trigger
robust_trim = False  # whether to trim outlier samples before the Rigol GMM plateau fit
ramp_step_size = 1.0  # maximum Avtech voltage change per ramp step, in volts
ramp_sleep_time = 2.0  # seconds to wait after each Avtech ramp step
idle_voltage = 5.0  # pulse voltage to ramp to once the sweep finishes normally
max_runtime_s = 1000.0  # safety cutoff: stop the sweep if it runs longer than this, in seconds
series_resistance_ohm = config.DEFAULT_SERIES_RESISTANCE_OHM  # known series/shunt resistance (R0)

print(
    f"mfli_n_samples={mfli_n_samples}, mfli_delay={mfli_delay}, "
    f"settle_time={settle_time}, robust_trim={robust_trim}, "
    f"ramp_step_size={ramp_step_size}, ramp_sleep_time={ramp_sleep_time}, "
    f"idle_voltage={idle_voltage}, max_runtime_s={max_runtime_s}, "
    f"series_resistance_ohm={series_resistance_ohm}"
)

## Live-plot callback

In [ ]:
def live_plot(df_so_far):
    """Redraw the I-V and intensity plots in place as the sweep progresses."""
    clear_output(wait=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    plotting.plot_iv_curve(df_so_far, ax=axes[0])
    plotting.plot_intensity_curve(df_so_far, ax=axes[1])
    plt.tight_layout()
    plt.show()

## Run the sweep and save the result, or fall back to synthetic dummy_data

This is the "one cell" step: it runs the sweep (live-plotting as it goes),
and always ends with `sweep_result` holding data + metadata, whether the
sweep completed, was interrupted, or timed out -- see `run_voltage_sweep`
in `measurement/sweep.py` for how interrupt/timeout safety is handled
internally.

In [ ]:
sweep_result = None

if instruments_connected:
    sweep_config = SweepConfig(
        voltage_channel=config.DEFAULT_CHANNEL_ROLES["voltage"],
        current_channel=config.DEFAULT_CHANNEL_ROLES["current"],
        series_resistance_ohm=series_resistance_ohm,
        ramp_step_size=ramp_step_size,
        ramp_sleep_time=ramp_sleep_time,
        settle_time=settle_time,
        robust_trim=robust_trim,
        mfli_n_samples=mfli_n_samples,
        mfli_delay=mfli_delay,
        idle_voltage=idle_voltage,
        max_runtime_s=max_runtime_s,
    )
    try:
        sweep_result = run_voltage_sweep(
            avtech,
            rigol,
            mfli,
            qdac,
            voltages=voltages,
            sweep_config=sweep_config,
            on_step=live_plot,
        )
        print(
            f"Sweep finished with status={sweep_result.status!r} "
            f"({sweep_result.completed_points}/{sweep_result.total_points} points)."
        )
    except Exception as exc:
        # Anything other than a plain interrupt (which run_voltage_sweep
        # already handles internally) -- e.g. a real communication error.
        print(f"Sweep failed unexpectedly: {exc}")

if sweep_result is None:
    # NOTE: hardware not available (or the sweep failed outright) -- this
    # is clearly-labeled synthetic dummy data, NOT a real measurement. It
    # loosely mimics a laser-like I-V/optical threshold: negligible
    # current/intensity below ~2 V, roughly linear current and intensity
    # above threshold.
    rng = np.random.default_rng(0)
    threshold_voltage = 2.0
    dut_voltage = 0.9 * voltages + rng.normal(0, 0.02, size=voltages.shape)
    above_threshold = np.clip(voltages - threshold_voltage, 0, None)
    dut_current = above_threshold * 0.01 + rng.normal(0, 0.0005, size=voltages.shape)
    lockin_x = above_threshold * 0.005 + rng.normal(0, 0.0002, size=voltages.shape)
    lockin_y = rng.normal(0, 0.0002, size=voltages.shape)
    lockin_r = np.hypot(lockin_x, lockin_y)
    lockin_phase = np.arctan2(lockin_y, lockin_x)

    # Synthetic std columns (same order of magnitude as the noise added
    # above), just so the schema matches a real sweep result exactly.
    dut_voltage_std = np.full_like(voltages, 0.02)
    dut_current_std = np.full_like(voltages, 0.0005)
    lockin_x_std = np.full_like(voltages, 0.0002)
    lockin_y_std = np.full_like(voltages, 0.0002)
    lockin_r_std = np.full_like(voltages, 0.0002)

    dummy_data = pd.DataFrame(
        {
            "set_voltage": voltages,
            "dut_voltage": dut_voltage,
            "dut_voltage_std": dut_voltage_std,
            "dut_current": dut_current,
            "dut_current_std": dut_current_std,
            "lockin_x": lockin_x,
            "lockin_x_std": lockin_x_std,
            "lockin_y": lockin_y,
            "lockin_y_std": lockin_y_std,
            "lockin_r": lockin_r,
            "lockin_r_std": lockin_r_std,
            "lockin_phase": lockin_phase,
        }
    )
    print("Using synthetic dummy_data (NOT a real measurement).")
    now = datetime.now(timezone.utc).isoformat()
    sweep_result = SweepResult(
        data=dummy_data,
        start_time=now,
        end_time=now,
        status="synthetic_dummy_data",
        completed_points=len(voltages),
        total_points=len(voltages),
        sweep_config=SweepConfig(),
    )

output_dir = project_root / "data"
run_dir = data_io.save_sweep_result(sweep_result, output_dir)
print(f"Saved sweep result to {run_dir} (sweep.csv + metadata.json).")

sweep_result.data

## Reload the saved result to confirm the round-trip

In [ ]:
reloaded_df, reloaded_metadata = data_io.load_sweep_result(run_dir)
print("Reloaded metadata:", reloaded_metadata)
reloaded_df

## Plot I-V and optical-intensity-vs-voltage curves

In [ ]:
if sweep_result.status == "synthetic_dummy_data":
    print("NOTE: the plots below are based on synthetic dummy_data, not a real measurement.")
else:
    print(f"The plots below are based on a real sweep measurement (status={sweep_result.status!r}).")

In [ ]:
plotting.plot_iv_curve(sweep_result.data, show=True)

In [ ]:
plotting.plot_intensity_curve(sweep_result.data, show=True)

## Clean up and disconnect all instruments

If the sweep above was interrupted or timed out, the Avtech is already
ramped to 0 V with its output off (done internally by
`run_voltage_sweep`); calling `output_off()` again here is harmless. This
section is the final full teardown for the end of the notebook session,
not part of the per-interrupt safety handling.

In [ ]:
try:
    if instruments_connected:
        avtech.output_off()
        qdac.abort_square_wave(1)
        qdac.abort_square_wave(2)
        qdac.set_channel_voltage(1, 0.0)
        qdac.set_channel_voltage(2, 0.0)
        print("Avtech output disabled; QDac trigger/gate channels aborted and zeroed.")
except Exception as exc:
    print(f"Error during instrument cleanup: {exc}")

In [ ]:
for name, instrument in [("QDac", qdac), ("Avtech", avtech), ("Rigol", rigol), ("MFLI", mfli)]:
    try:
        if instruments_connected:
            instrument.disconnect()
            print(f"Disconnected from {name}.")
    except Exception as exc:
        print(f"Error disconnecting {name}: {exc}")

if not instruments_connected:
    print("Instruments were never connected; nothing to disconnect.")